In [ ]:
# Algorithms Session 6
# At this point, you’ve covered most major interview families, so Session 6 should be a mixed recognition session with slightly less obvious pattern cues.
# Today’s 5 problems cover:
# - trie
# - union-find / disjoint set
# - dynamic programming on strings
# - topological sort
# - monotonic queue / deque

In [ ]:
# Problem 1 — Implement Trie
# Pattern: Trie / prefix tree
# Implement a data structure that supports:
# - inserting a word
# - checking whether a complete word exists
# - checking whether any word starts with a given prefix

#Logic
# 1. Create a TrieNode class
# 2. Each node stores:
#    - children: a dictionary of next characters
#    - is_end: whether a complete word ends here
# 3. Trie starts with an empty root node
# 4. insert(word):
#    - start at root
#    - for each character:
#      create a child node if it does not exist
#      move to that child
#    - mark the final node as end of a word
# 5. search(word):
#    - follow each character from the root
#    - if a character is missing, return False
#    - after all characters, return True only if is_end is True
# 6. starts_with(prefix):
#    - follow each character from the root
#    - if a character is missing, return False
#    - if all prefix characters exist, return True

class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        current = self.root

        for char in word:
            if char not in current.children:
                current.children[char] = TrieNode()

            current = current.children[char]

        current.is_end = True

    def search(self, word):
        current = self.root

        for char in word:
            if char not in current.children:
                return False

            current = current.children[char]

        return current.is_end

    def starts_with(self, prefix):
        current = self.root

        for char in prefix:
            if char not in current.children:
                return False

            current = current.children[char]

        return True


trie = Trie()

trie.insert("apple")

print(trie.search("apple"))       # True
print(trie.search("app"))         # False
print(trie.starts_with("app"))    # True

trie.insert("app")

print(trie.search("app"))         # True

In [ ]:
# Problem 2 — Number of Connected Components
# Pattern: Union-Find / Disjoint Set

#Logic 
# 1. Start with each node as its own parent
# 2. Start the component count at n
# 3. find(x):
#    - follow parent links until you reach the root
#    - use path compression so future finds are faster
# 4. union(a, b):
#    - find the root of a
#    - find the root of b
# 5. If both nodes already have the same root,
#    they are already connected, so do nothing
# 6. Otherwise, connect the smaller-rank tree
#    under the larger-rank tree
# 7. If both ranks are equal,
#    choose one root and increase its rank by 1
# 8. Every successful union reduces
#    the component count by 1
# 9. Process every edge
# 10. Return the final component count

def count_components(n, edges):
    parent = list(range(n))
    rank = [0] * n
    count = n

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])

        return parent[x]

    def union(a, b):
        nonlocal count

        root_a = find(a)
        root_b = find(b)

        if root_a == root_b:
            return

        if rank[root_a] < rank[root_b]:
            parent[root_a] = root_b

        elif rank[root_a] > rank[root_b]:
            parent[root_b] = root_a

        else:
            parent[root_b] = root_a
            rank[root_a] += 1

        count -= 1

    for a, b in edges:
        union(a, b)

    return count


n = 5
edges = [
    [0, 1],
    [1, 2],
    [3, 4]
]
print(count_components(n, edges))  # 2



In [ ]:
# Problem 3 — Longest Common Subsequence
# Pattern: Dynamic programming
# Given two strings text1 and text2, return the length of their longest common subsequence.
# A subsequence does not need to be contiguous.

#Logic
# 1. Create a 2D DP table with:
#    rows = len(text1) + 1
#    cols = len(text2) + 1
# 2. dp[i][j] represents the LCS length between:
#    text1[:i] and text2[:j]
# 3. Go through both strings character by character
# 4. If text1[i - 1] == text2[j - 1]:
#    take the diagonal value and add 1
# 5. Otherwise:
#    take the maximum of:
#    - ignoring the current character from text1
#    - ignoring the current character from text2
# 6. The final answer is in dp[len(text1)][len(text2)]
# 7. Return that value

def longest_common_subsequence(text1, text2):
    m = len(text1)
    n = len(text2)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = 1 + dp[i - 1][j - 1]

            else:
                dp[i][j] = max(
                    dp[i - 1][j],
                    dp[i][j - 1]
                )

    return dp[m][n]

print(longest_common_subsequence("abcde", "ace"))  # 3
print(longest_common_subsequence("abc", "abc"))    # 3
print(longest_common_subsequence("abc", "def"))    # 0

In [ ]:
# Problem 4 — Course Schedule
# Pattern: Topological sort / cycle detection

#Logic
# 1. Build a graph showing which courses depend on each course
# 2. Create an indegree list
#    indegree[i] = number of prerequisites course i still needs
# 3. For each prerequisite [a, b]:
#    add an edge b -> a
#    increase indegree[a] by 1
# 4. Put every course with indegree 0 into a queue
#    because those courses can be taken immediately
# 5. Remove courses from the queue one by one
# 6. For each course removed:
#    increase processed count
#    reduce the indegree of its dependent courses
# 7. If a dependent course reaches indegree 0,
#    add it to the queue
# 8. At the end, if processed == num_courses,
#    all courses can be finished → True
# 9. Otherwise, some courses are stuck in a cycle → False

from collections import deque

def can_finish(num_courses, prerequisites):
    graph = [[] for _ in range(num_courses)]
    indegree = [0] * num_courses

    for course, prerequisite in prerequisites:
        graph[prerequisite].append(course)
        indegree[course] += 1

    queue = deque()

    for course in range(num_courses):
        if indegree[course] == 0:
            queue.append(course)

    processed = 0

    while queue:
        current = queue.popleft()
        processed += 1

        for next_course in graph[current]:
            indegree[next_course] -= 1

            if indegree[next_course] == 0:
                queue.append(next_course)

    return processed == num_courses


print(can_finish(2, [[1, 0]]))
# True
print(can_finish(2, [[1, 0], [0, 1]]))
# False

In [ ]:
# Problem 5 — Sliding Window Maximum
# Pattern: Monotonic deque
# Given an array nums and integer k, return the maximum value in every contiguous window of size k.

#Logic
# 1. Create an empty deque to store indices
# 2. Create an empty result list
# 3. Go through nums from left to right
# 4. Remove indices from the front if they are
#    outside the current window
# 5. Remove indices from the back while their values
#    are smaller than or equal to the current value
# 6. Add the current index to the deque
# 7. Once the first full window of size k is formed,
#    nums[deque[0]] is the maximum for that window
# 8. Add that maximum to the result
# 9. Return the result

from collections import deque

def max_sliding_window(nums, k):
    dq = deque()
    result = []

    for i in range(len(nums)):

        # Remove indices outside the current window
        if dq and dq[0] <= i - k:
            dq.popleft()

        # Remove smaller values from the back
        while dq and nums[dq[-1]] <= nums[i]:
            dq.pop()

        # Add current index
        dq.append(i)

        # Start recording answers once window size reaches k
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result


print(max_sliding_window(
    [1, 3, -1, -3, 5, 3, 6, 7],
    3
))
# [3, 3, 5, 5, 6, 7]